In [ ]:
import pandas as pd
import numpy as np


settings_file = "kinematic_settings.csv"  
bigtable_file = "../rsidis_bigtable_pass0.csv"

fit_files = {
    "H_cer": "H.cer.npeSum_fit_results_all.csv",
    "P_hgcer": "P.hgcer.npeSum_fit_results_all.csv",
    "P_aero": "P.aero.npeSum_fit_results_all.csv",
}
settings = pd.read_csv(settings_file)
bigtable = pd.read_csv(bigtable_file)
bigtable["run"] = bigtable["run"].astype(int)

# Load fit files
fit_data = {}
for det, file in fit_files.items():
    df = pd.read_csv(file)
    df["run"] = df["run"].astype(int)
    # Ensure chi2_ndf exists
    df["chi2_ndf"] = df["chi2"] / df["ndf"]
    # Compute mean_err
    df["mean_err"] = df["std_dev"] / np.sqrt(df["nevents"].clip(lower=1))
    fit_data[det] = df


def is_good_fit(row):
    chi2ndf = row["chi2_ndf"]
    nevents = row["nevents"]
    return (0.5 < chi2ndf) and (nevents > 83)


def compute_weighted_mean(df):
    if df.empty:
        return np.nan, np.nan

    df = df[df["mean_err"] > 0]
    if df.empty:
        return np.nan, np.nan

    w = 1 / df["mean_err"]**2
    wm = np.sum(df["mean"] * w) / np.sum(w)
    wstd = np.sqrt(np.sum(w * (df["mean"] - wm)**2) / np.sum(w))
    wm_err = 1 / np.sqrt(np.sum(w))

    return wm, wstd, wm_err


settings["run_type"] = settings["polarity"].map({
    -1: "PI-SIDIS",
    +1: "PI+SIDIS"
})


output = []

for idx, row in settings.iterrows():
    ebeam = row["ebeam"]
    x = row["x"]
    Q2 = row["Q2"]
    z = row["z"]
    thpq = row["thpq"]
    run_type = row["run_type"]

    print(f"\nProcessing setting {idx+1}/{len(settings)}:")
    print(f"  {run_type}, E={ebeam}, x={x}, Q2={Q2}, z={z}, thpq={thpq}")

    # Filter runs
    mask = (
        (bigtable["run_type"] == run_type)
        & (bigtable["ebeam"] == ebeam)
        & (bigtable["x"] == x)
        & (bigtable["Q2"] == Q2)
        & (bigtable["z"] == z)
        & (bigtable["thpq"] == thpq)
        & (bigtable["hms_p"] < 0)  
    )

    matched_runs = bigtable[mask]
    
    if matched_runs.empty:
        print("  ⚠ No matching runs found")
        result = {
            "polarity": row["polarity"],
            "ebeam": ebeam, "x": x, "Q2": Q2, "z": z, "thpq": thpq,
            "H_cer_mean": np.nan, "H_cer_std": np.nan,
            "P_hgcer_mean": np.nan, "P_hgcer_std": np.nan,
            "P_aero_mean": np.nan, "P_aero_std": np.nan,
        }
        output.append(result)
        continue

    result = {
        "polarity": row["polarity"],
        "ebeam": ebeam, "x": x, "Q2": Q2, "z": z, "thpq": thpq,
    }

    for det_name, det_df in fit_data.items():
        merged = pd.merge(
            matched_runs[["run", "target"]],
            det_df,
            on="run",
            how="inner"
        )

        good = merged[merged.apply(is_good_fit, axis=1)]

        w_mean, w_std, w_err = compute_weighted_mean(good)

        result[f"{det_name}_mean"] = w_mean
        result[f"{det_name}_std"]  = w_std
        result[f"{det_name}_err"]  = w_err

        if np.isnan(w_mean):
            print(f"  {det_name}: no good fits")
        else:
            print(f"  {det_name}: mean={w_mean:.3f}, std={w_std:.3f}, err={w_err:.3f}")

    output.append(result)


output_df = pd.DataFrame(output)
output_df.to_csv("all_particle_means.csv", index=False)

print("\nDONE! Saved → all_particle_means.csv")



Processing setting 1/27:
  PI-SIDIS, E=8.5831, x=0.25, Q2=3.3, z=0.36, thpq=2.0
  H_cer: mean=9.229, std=0.428, err=0.089
  P_hgcer: mean=8.053, std=0.000, err=0.364
  P_aero: mean=7.959, std=0.000, err=0.326

Processing setting 2/27:
  PI-SIDIS, E=8.5831, x=0.25, Q2=3.3, z=0.5, thpq=2.0
  H_cer: mean=8.884, std=0.165, err=0.008
  P_hgcer: mean=13.264, std=0.900, err=0.035
  P_aero: mean=8.247, std=0.364, err=0.017

Processing setting 3/27:
  PI-SIDIS, E=8.5831, x=0.25, Q2=3.3, z=0.67, thpq=2.0
  H_cer: mean=8.894, std=0.165, err=0.008
  P_hgcer: mean=23.139, std=1.698, err=0.045
  P_aero: mean=8.794, std=0.390, err=0.018

Processing setting 4/27:
  PI-SIDIS, E=8.5831, x=0.25, Q2=3.3, z=0.9, thpq=2.0
  H_cer: mean=8.866, std=0.206, err=0.025
  P_hgcer: mean=28.516, std=1.097, err=0.149
  P_aero: mean=9.348, std=0.302, err=0.057

Processing setting 5/27:
  PI-SIDIS, E=8.5831, x=0.25, Q2=3.3, z=0.5, thpq=5.2
  H_cer: mean=8.928, std=0.146, err=0.011
  P_hgcer: mean=10.426, std=0.786, er